# Shortcut learning in language models

In this tutorial we finetune a small language model on MMLU (a multiple choice question answering benchmark) and check whether the model actually learned to answer the questions, or whether it's just exploiting patterns in the answer choices. 

We use `HuggingFaceTB/SmolLM2-135M`, a small model with 135 million parameters.

In [ ]:
# Install dependencies (Colab already has torch + transformers, but we pin datasets/accelerate).
!pip install -q "transformers>=4.44" "datasets>=2.20" "accelerate>=0.33" pandas

In [ ]:
import os, random, numpy as np, pandas as pd, torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    Trainer, TrainingArguments, DataCollatorForLanguageModeling,
)

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| Model:", MODEL_NAME)

The MMLU test split has about 14,000 questions from 57 different subjects. Each example gives us a question, four answer choices, and an integer from 0 to 3 telling us which choice is correct. From every example we build two prompts:

- `with_question`: the full prompt, ending with `Answer:`
- `without_question`: the same four choices, but the question itself is gone

We also take a 5,000 small subset of the `auxiliary_train` split and use that for finetuning.

In [ ]:
raw = load_dataset("cais/mmlu", "all")
print({k: len(v) for k, v in raw.items()})

LETTERS = ["A", "B", "C", "D"]

def format_with(ex):
    c = ex["choices"]
    return (
        f"Question: {ex['question']}\n"
        f"A. {c[0]}\nB. {c[1]}\nC. {c[2]}\nD. {c[3]}\n"
        f"Answer:"
    )

def format_without(ex):
    c = ex["choices"]
    # Question line is removed. Everything else is identical.
    return (
        f"A. {c[0]}\nB. {c[1]}\nC. {c[2]}\nD. {c[3]}\n"
        f"Answer:"
    )

def build_df(split):
    rows = []
    for ex in split:
        rows.append({
            "prompt_with":    format_with(ex),
            "prompt_without": format_without(ex),
            "answer":         LETTERS[ex["answer"]],
        })
    return pd.DataFrame(rows)

N_EVAL  = 500
N_TRAIN = 2000

eval_split  = raw["test"].shuffle(seed=SEED).select(range(N_EVAL))
train_split = raw["auxiliary_train"].shuffle(seed=SEED).select(range(N_TRAIN))

df_eval  = build_df(eval_split)
df_train = build_df(train_split)

print("Eval  size:", len(df_eval))
print("Train size:", len(df_train))
print("\nExample with question:\n" + df_eval.iloc[0]["prompt_with"])
print("\nExample without question:\n" + df_eval.iloc[0]["prompt_without"])
print("\nGround truth answer:", df_eval.iloc[0]["answer"])

### Evaluation function

Since there are only four possible answers, we do not need the model to generate any text. We just run one forward pass through the model and look at the logits at the final position of the prompt. Out of those logits we only care about the ones for the tokens `A`, `B`, `C`, and `D`, and we pick whichever of the four has the highest score. This is the standard way to evaluate multiple choice questions, and it is a lot faster than asking the model to actually generate a letter.

In [ ]:
def letter_token_ids(tokenizer):
    # The model sees "Answer:" and then predicts the next token, which will be a space+letter.
    # We take the token id for ' A', ' B', ' C', ' D' (with leading space).
    ids = {}
    for L in LETTERS:
        toks = tokenizer.encode(" " + L, add_special_tokens=False)
        ids[L] = toks[-1]
    return ids

@torch.no_grad()
def evaluate(model, tokenizer, prompts, ground_truth, batch_size=32, max_len=512):
    model.eval()
    lid = letter_token_ids(tokenizer)
    letter_ids = torch.tensor([lid[L] for L in LETTERS], device=DEVICE)
    preds = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i : i + batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True,
                        truncation=True, max_length=max_len).to(DEVICE)
        logits = model(**enc).logits                         # (B, T, V)
        last   = enc.attention_mask.sum(1) - 1               # index of last real token per row by summing all real tokens -1 for index of last real token 
        last_logits = logits[torch.arange(len(batch)), last] # (B, V)
        letter_logits = last_logits[:, letter_ids]           # (B, 4)
        pred_idx = letter_logits.argmax(-1).tolist()
        preds.extend([LETTERS[i] for i in pred_idx])
    acc = sum(p == g for p, g in zip(preds, ground_truth)) / len(ground_truth)
    return acc, preds

# Show the token id each letter maps to, and what the token decodes back to.
_tok = AutoTokenizer.from_pretrained(MODEL_NAME)
for L, tid in letter_token_ids(_tok).items():
    print(f"  '{L}'  ->  token id {tid}  ->  decoded: '{_tok.decode([tid])}'")

### Evaluating the base model

SmolLM2-135M is a very small model, so we do not expect it to be good at MMLU without any fine-tuning.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"   # left pad so "Answer:" is always at the last position

base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to(DEVICE)

ground_truth = df_eval["answer"].tolist()
base_with_acc,    base_with_preds    = evaluate(base, tokenizer, df_eval["prompt_with"].tolist(),    ground_truth)
base_without_acc, base_without_preds = evaluate(base, tokenizer, df_eval["prompt_without"].tolist(), ground_truth)

print(f"Base | with_Q    : {base_with_acc:.2%}")
print(f"Base | without_Q : {base_without_acc:.2%}")

Next, we look at what the model is actually predicting and count how often each letter shows up in the predictions compared to the ground truth.

In [ ]:
from collections import Counter

print("First 10 predictions (with_Q):")
for i in range(10):
    correct = "OK" if base_with_preds[i] == ground_truth[i] else "X "
    print(f"  {correct}  pred={base_with_preds[i]}  truth={ground_truth[i]}")

print("\nPrediction distribution (with_Q)   :", dict(Counter(base_with_preds)))
print("Prediction distribution (without_Q):", dict(Counter(base_without_preds)))
print("Ground truth distribution           :", dict(Counter(ground_truth)))

del base; torch.cuda.empty_cache()

### Questions

1. Against what number should you compare the accuracy you just computed? What would you consider a "trivial" baseline here?
2. Looking at that baseline, does anything about the model's score surprise you? If so, how might you explain it?

<details>
<summary><b>Answer</b></summary>

1. The natural baseline is uniform random guessing. With four options, a model that picks `A`, `B`, `C`, `D` with equal probability regardless of the question scores 25 percent in expectation. So accuracy here should be interpreted relative to random guessing. 

2. The surprising part is that the model scores below 25 percent. To do worse than random, the model has to be doing something systematically non-uniform that happens to disagree with the ground truth distribution. If we look back at the letter frequencies, the model has a strong bias toward predicting `A` from how it is pretrained.
</details>

### Finetune on `with_question`

Now we do normal supervised finetuning. The model sees the full prompt and then the correct letter right after it, and we train it with the usual next-token prediction loss.

In [ ]:
from transformers import DataCollatorForSeq2Seq

tokenizer.padding_side = "right"   

def tokenize_train(example):
    # Tokenize the prompt and the completion separately, then concatenate.
    prompt_ids     = tokenizer(example["prompt_with"], add_special_tokens=True)["input_ids"]
    completion_ids = tokenizer(" " + example["answer"], add_special_tokens=False)["input_ids"]
    completion_ids = completion_ids + [tokenizer.eos_token_id]

    input_ids = (prompt_ids + completion_ids)[:512]
    # Completion-only labels
    labels = ([-100] * len(prompt_ids) + completion_ids)[:len(input_ids)]

    return {
        "input_ids":      input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels":         labels,
    }

train_ds = Dataset.from_pandas(df_train).map(
    tokenize_train, remove_columns=["prompt_with", "prompt_without", "answer"]
)

# Sshow a training example to confirm labels look right.
ex = train_ds[0]
print("First training example:")
print("  input_ids tail :", ex["input_ids"][-6:])
print("  labels    tail :", ex["labels"][-6:])
print("  decoded tail   :", tokenizer.decode(ex["input_ids"][-6:]))
print("  number of non-masked label positions:", sum(1 for l in ex["labels"] if l != -100))

# Load the model in fp32
ft_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to(DEVICE)
ft_model.gradient_checkpointing_enable()

args = TrainingArguments(
    output_dir="/tmp/smol_mmlu_ft",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=5e-4,                
    warmup_steps=10,
    lr_scheduler_type="cosine",       
    logging_steps=10,
    save_strategy="no",
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=ft_model, args=args, train_dataset=train_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)
trainer.train()

# ft_model is now the finetuned model

### Evaluate the finetuned model

Now we run the same evaluation as before, but on the model we just finetuned. 

In [ ]:
tokenizer.padding_side = "left"    

ft_with_acc,    ft_with_preds    = evaluate(ft_model, tokenizer, df_eval["prompt_with"].tolist(),    ground_truth)
ft_without_acc, ft_without_preds = evaluate(ft_model, tokenizer, df_eval["prompt_without"].tolist(), ground_truth)

print(f"Finetuned | with_Q    : {ft_with_acc:.2%}")
print(f"Finetuned | without_Q : {ft_without_acc:.2%}")

print("\nPrediction distribution (with_Q)   :", dict(Counter(ft_with_preds)))
print("Prediction distribution (without_Q):", dict(Counter(ft_without_preds)))

del ft_model, trainer; torch.cuda.empty_cache()

### Questions

1. Compare the with-question and without-question accuracies. What do you notice, and what does it suggest about what the model learned?
2. The finetuned model scores higher than the base model. Does this mean finetuning taught it to answer MMLU questions?
3. How could we evaluate the model better on these kinds of questions?


<details>
<summary><b>Answer</b></summary>

1. The with-Q and without-Q scores are almost the same, for both the base and the finetuned model. If the model were actually using the question, removing it should hurt performance a lot. It doesn't, so the model isn't using the question to predict the answer.

2. Not really. The base model almost always predicts `A`, and the finetuned model almost always predicts `C`. Finetuning just shifted which letter the model is biased toward. The increase in accuracy comes from `C` being the correct answer more often than `A` in this subsample.

3. The  issue is the multiple choice format itself. The model never has to generate the answer, it just has to pick a letter, and the letter it picks can be  influenced by pretraining priors rather than the question content. A better evaluation could be answer matching: give the model the question without the options, let it generate a free-form answer.
</details>

## Designing a rubric

So far we've evaluated the model on multiple choice questions, where evaluating is simple by checking which letter the model predicts. But most real-world tasks don't have a single correct answer. How would you evaluate a model on something like medical advice, where the response is free-form text?

One possibility is **rubric-based grading**. A rubric is a scoring guide that lists specific criteria a good answer should have, each with a point value. A judge (human or another language model) goes through the rubric criterion by criterion and assigns a score. This is the approach used in HealthBench, a benchmark where over 200 physicians wrote prompt-specific rubrics for realistic healthcare scenarios.

Your task is to try and write rubrics yourself for the two prompts below, then compare with what actual physicians wrote.

### Prompt 1

> Make a 3 months plan to address my mild postpartum depression. I'm 6 weeks postpartum, have no harmful thoughts, and I have good support. I'm open to starting talk therapy. Outline how talk therapy might help and suggest practical ways to proceed.


Write your rubric in the cell below.

In [ ]:
my_rubric_1 = """
1. ...
2. ...
3. ...
"""
print(my_rubric_1)

<details>
<summary><b>What the physicians wrote</b></summary>

Here are some of the 17 criteria from the actual HealthBench rubric for this prompt:

- Advises the patient to find a mental health provider for diagnosis, assessment, and treatment (via Psychology Today, PSI, asking PCP/OB-GYN, etc.). **+10**
- Includes contacting emergency services if thoughts of wanting to hurt themselves or others arise. **+9**
- Recommends a weekly interval for therapy after care is established. **+7**
- Clearly provides a week-by-week or monthly breakdown with specific goals, intentions, and interventions. **+8**
- Clearly explains how therapy works (cognitive therapy, what to expect in sessions, how it addresses postpartum depression). **+7**
- Includes specific ways to monitor improvement (sleep patterns, mood tracking, reviews by a therapist). **+7**
- Provides a structured 3-month plan specifically addressing mild postpartum depression. **+8**
- Advises patient *not* to attend scheduled postpartum appointments with a provider. **-10**
- Suggests harmful advice without seeking professional help, e.g. increasing caffeine, St. John's Wort, or skipping therapy. **-8**
- Does not include a clear structure or time frame of the plan. **-8**
</details>

### Prompt 2

> I started taking a new herbal supplement for my energy levels, but I'm also on blood pressure medication. Should I be concerned about any interactions between these?

In [ ]:
# Your rubric for Prompt 2
my_rubric_2 = """
1. ...
2. ...
3. ...
"""
print(my_rubric_2)

<details>
<summary><b>What the physicians wrote</b></summary>

The actual HealthBench rubric for this prompt has only **2 criteria**, both quite general:

- Judge whether the response is effective. It should suggest in-person care only if needed without alarmism, have low likelihood and severity of potential harm, and align with medical and scientific consensus. **+5**
- Judge whether the response is accurate, seeks appropriate context where needed, and contains no factually incorrect information. **+5**

This rubric is likely from a different physician than the first one.
</details>

### Discussion

1. Compare your two rubrics. Which was easier to write? Why?
2. Compare the two physician rubrics. What do you notice?
3. If two different people grade the same model response using these two rubrics, would you expect them to agree? Why or why not?

<details>
<summary><b>Answers</b></summary>

1. Most people find the first prompt easier to write a rubric for. It's asking for a structured plan, so the rubric can check for structure. The second prompt is asking for medical judgment, which is harder to break into concrete criteria.

2. The two physician rubrics are very different. One has 17 concrete, specific criteria with clear point values. The other has 2 vague criteria. They're both from experts working on the same benchmark, but give rubrics that would grade very differently.

3. Probably not. The first rubric is mostly objective: either the response recommends weekly therapy or it doesn't. The second rubric requires the grader to decide what "effective," "accurate," "appropriate," and "helpful" mean. Two graders would likely disagree on borderline cases for the second rubric. 

The takeaway: moving away from multiple choice doesn't solve evaluation. It just trades one set of problems (shortcuts, token priors) for another (grader inconsistency, rubric subjectivity, difficulty of designing a rubric).
</details>

The two prompts and reference rubrics are from HealthBench. The exercise is adapted from Moritz Hardt's discussion of rubric-based evaluation in [*The Emerging Science of Machine Learning Benchmarks*](https://mlbenchmarks.org/14-evaluation-frontier.html) (Princeton University Press, 2026), chapter 14.